# 04 - ECAPA-TDNN Speaker Verification

## Objective

Evaluate the ability of ECAPA-TDNN embeddings to distinguish whether two utterances belong to the same speaker.

We construct speaker verification trials from 100 selected speakers:

- **Positive trials:** two utterances from the same speaker, taken from different videos.
- **Negative trials:** two utterances from different speakers.

For each pair of utterances $(X_1,X_2)$, we compute their ECAPA embeddings and use cosine similarity as the verification score:

$$
s(X_1,X_2)
=
\cos\left(
Z_{\mathrm{id}}(X_1),
Z_{\mathrm{id}}(X_2)
\right).
$$

A higher similarity score indicates greater evidence that the two utterances belong to the same speaker.

The trials are evaluated using the ROC curve and Equal Error Rate (EER), together with the corresponding threshold, False Acceptance Rate (FAR), and False Rejection Rate (FRR).

This provides a quantitative baseline for the amount of speaker identity information captured by the ECAPA representation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_curve

import sys

In [2]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        parent
        for parent in (cwd, *cwd.parents)
        if (parent / "src").is_dir() and (parent / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the repository root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paths import (
    VOXCELEB_WAV_DIR,
    METADATA_DIR,
    PROCESSED_DATA_DIR,
)

TRIALS_DIR = PROJECT_ROOT / "data" / "trials"

print("Project root:", PROJECT_ROOT)
print("VoxCeleb wav:", VOXCELEB_WAV_DIR)
print("Metadata directory exists:", METADATA_DIR.is_dir())

Project root: /Users/juni/Documents/interview-anonymization
VoxCeleb wav: /Users/juni/Documents/interview-anonymization/data/raw/voxceleb1/wav
Metadata directory exists: True


In [3]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.attacker_baseline.ecapa import load_ecapa, extract_embedding

classifier = load_ecapa()

In [4]:
utterance_df = pd.read_parquet(
    METADATA_DIR / "voxceleb_utterances.parquet"
)

print("Utterances:", len(utterance_df))
print("Speakers:", utterance_df["speaker_id"].nunique())

utterance_df.head()

Utterances: 124843
Speakers: 1166


,speaker_id,video_id,utterance_id,path,split,language,language_confidence
0,id10001,1zcIwhmdeo4,00001,id10001/1zcIwhmdeo4/00001.wav,train,english,0.99
1,id10001,1zcIwhmdeo4,00002,id10001/1zcIwhmdeo4/00002.wav,train,english,0.99
2,id10001,1zcIwhmdeo4,00003,id10001/1zcIwhmdeo4/00003.wav,train,english,0.99
3,id10001,7gWzIy6yIIk,00001,id10001/7gWzIy6yIIk/00001.wav,train,english,0.98
4,id10001,7gWzIy6yIIk,00002,id10001/7gWzIy6yIIk/00002.wav,train,english,0.98


In [5]:
speaker_stats = (
    utterance_df.groupby("speaker_id")["video_id"]
    .nunique()
    .rename("n_videos")
    .to_frame()
)

eligible_speakers = speaker_stats.loc[
    speaker_stats["n_videos"] >= 5
].index.tolist()

print("Eligible speakers:", len(eligible_speakers))

Eligible speakers: 1132


In [6]:
import random

SEED = 42
N_SPEAKERS = 100
POS_PER_SPEAKER = 10
NEG_PER_SPEAKER = 10

rng = random.Random(SEED)

selected_speakers = rng.sample(
    eligible_speakers,
    N_SPEAKERS
)

len(selected_speakers)

100

In [7]:
positive_trials = []

for speaker in selected_speakers:

    speaker_df = utterance_df[
        utterance_df["speaker_id"] == speaker
    ]

    videos = speaker_df["video_id"].unique()

    for _ in range(POS_PER_SPEAKER):

        video1, video2 = rng.sample(list(videos), 2)

        utt1 = speaker_df[
            speaker_df["video_id"] == video1
        ].sample(
            1,
            random_state=rng.randint(0, 1_000_000)
        ).iloc[0]

        utt2 = speaker_df[
            speaker_df["video_id"] == video2
        ].sample(
            1,
            random_state=rng.randint(0, 1_000_000)
        ).iloc[0]

        positive_trials.append({
            "label": 1,
            "path1": utt1["path"],
            "path2": utt2["path"],
            "speaker1": speaker,
            "speaker2": speaker
        })

len(positive_trials)

1000

In [8]:
negative_trials = []

for _ in range(N_SPEAKERS * NEG_PER_SPEAKER):

    speaker1, speaker2 = rng.sample(
        selected_speakers,
        2
    )

    utt1 = utterance_df[
        utterance_df["speaker_id"] == speaker1
    ].sample(
        1,
        random_state=rng.randint(0, 1_000_000)
    ).iloc[0]

    utt2 = utterance_df[
        utterance_df["speaker_id"] == speaker2
    ].sample(
        1,
        random_state=rng.randint(0, 1_000_000)
    ).iloc[0]

    negative_trials.append({
        "label": 0,
        "path1": utt1["path"],
        "path2": utt2["path"],
        "speaker1": speaker1,
        "speaker2": speaker2
    })

len(negative_trials)

1000

In [9]:
verification_trials = pd.DataFrame(
    positive_trials + negative_trials
)

verification_trials = verification_trials.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print(verification_trials.shape)
print()
print(verification_trials["label"].value_counts())

verification_trials.head()

(2000, 5)

label
0    1000
1    1000
Name: count, dtype: int64


,label,path1,path2,speaker1,speaker2
0,0,id10819/aAfmE3MWsrE/00001.wav,id10400/bujquo47jac/00003.wav,id10819,id10400
1,1,id10095/kZ10dA7AmR0/00002.wav,id10095/E7inQ8WDwnM/00002.wav,id10095,id10095
2,0,id10203/22uUS8vq48c/00028.wav,id10877/c_rgZKwfb5s/00003.wav,id10203,id10877
3,1,id10198/RwgzosJzMmo/00001.wav,id10198/EpPA1Mf32Mk/00005.wav,id10198,id10198
4,0,id10540/5PLMVr9rFeY/00006.wav,id10058/MV4B_V28-j8/00004.wav,id10540,id10058


In [10]:
TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
TRIALS_DIR.mkdir(parents=True, exist_ok=True)

OUR_TRIALS_PATH = (
    TRIALS_DIR / "voxceleb_verification_trials.csv"
)

verification_trials.to_csv(
    OUR_TRIALS_PATH,
    index=False
)

print("Saved:", OUR_TRIALS_PATH)

Saved: /Users/juni/Documents/interview-anonymization/data/trials/voxceleb_verification_trials.csv


In [11]:
unique_paths = sorted(
    set(verification_trials["path1"])
    | set(verification_trials["path2"])
)

print("Trials:", len(verification_trials))
print("Unique utterances:", len(unique_paths))

Trials: 2000
Unique utterances: 3100


In [12]:
import paths

print("Module chargé depuis :", paths.__file__)
print("Project root :", paths.PROJECT_ROOT)
print("Audio directory :", paths.VOXCELEB_WAV_DIR)

Module chargé depuis : /Users/juni/Documents/interview-anonymization/src/paths.py
Project root : /Users/juni/Documents/interview-anonymization
Audio directory : /Users/juni/Documents/interview-anonymization/data/raw/voxceleb1/wav


In [13]:
from tqdm import tqdm

embedding_cache = {}

unique_paths = [
    VOXCELEB_WAV_DIR / path
    for path in unique_paths
]

for path in tqdm(unique_paths):
    embedding_cache[path] = extract_embedding(
        path,
        classifier
    )

print("Embeddings computed:", len(embedding_cache))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3100/3100 [06:42<00:00,  7.71it/s]

Embeddings computed: 3100


In [14]:
def cosine_similarity(z1, z2):
    z1 = np.asarray(z1).reshape(-1)
    z2 = np.asarray(z2).reshape(-1)

    z1 = z1 / np.linalg.norm(z1)
    z2 = z2 / np.linalg.norm(z2)

    return np.dot(z1, z2)

In [15]:
scores = []

embedding_cache = {
    Path(path).relative_to(VOXCELEB_WAV_DIR).as_posix(): embedding
    for path, embedding in embedding_cache.items()
}

for _, row in verification_trials.iterrows():
    z1 = embedding_cache[row["path1"]]
    z2 = embedding_cache[row["path2"]]

    score = cosine_similarity(z1, z2)
    scores.append(score)

verification_trials["score"] = scores

verification_trials[
    ["label", "speaker1", "speaker2", "score"]
].head(10)

,label,speaker1,speaker2,score
0,0,id10819,id10400,0.001716
1,1,id10095,id10095,0.635017
2,0,id10203,id10877,0.112272
3,1,id10198,id10198,0.769848
4,0,id10540,id10058,0.000125
5,0,id10972,id10634,0.001138
6,1,id10375,id10375,0.614294
7,0,id10068,id10500,0.360134
8,1,id10222,id10222,0.582467
9,0,id11159,id10401,-0.046516


In [16]:
verification_trials.groupby("label")["score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,1000.0,0.006863,0.091751,-0.277980,-0.052070,0.005504,0.062523,0.363198
1,1000.0,0.624391,0.116312,0.079603,0.553734,0.635148,0.704242,0.976121


In [17]:
from sklearn.metrics import roc_curve

labels = verification_trials["label"].to_numpy()
scores = verification_trials["score"].to_numpy()

fpr, tpr, thresholds = roc_curve(labels, scores)

fnr = 1 - tpr

eer_idx = np.nanargmin(np.abs(fpr - fnr))

eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold = thresholds[eer_idx]

print(f"EER: {eer * 100:.20f}%")
print(f"EER threshold: {eer_threshold:.3f}")
print(f"FAR at EER: {fpr[eer_idx] * 100:.2f}%")
print(f"FRR at EER: {fnr[eer_idx] * 100:.2f}%")

EER: 0.65000000000000035527%
EER threshold: 0.278
FAR at EER: 0.50%
FRR at EER: 0.80%
